In [ ]:
from utils.visualization import *
from geometry.importers import STEPImporter
from solvers.frequency_domain import FrequencyDomainSolver
from rom.reduction import ModelOrderReduction
from analytical.rectangular_waveguide import RWGAnalytical
from analytical.cst_result import CSTResult
from ngsolve.webgui import Draw # must import Draw, otherwise may run into problems showing mesh
from utils.visualization import (
    plot_z_comparison, 
    plot_s_comparison, 
    plot_all_parameters,
    plot_eigenfrequencies,
    ParameterPlotter,
    EigenfrequencyPlotter
)
%matplotlib widget
plt.rcParams['figure.dpi'] = 100

## 1. Define Geometry

Create a circular waveguide with specified dimensions and mesh parameters.

In [ ]:
class Models:
    def rwg(self, maxh=0.04):
        geo = STEPImporter(r"../rwg/rectangular_waveguide.step", auto_build=False)
        geo.build()
        geo.name_solids()
        geo.generate_mesh(maxh)
        return geo
    def rwg_split(self, maxh=0.04):
        geo = STEPImporter(r"../rwg/rectangular_waveguide.step", auto_build=False)
        geo.add_splitting_plane_at_z(0.05)
        geo.add_splitting_plane_at_z(0.10)
        geo.split()
        geo.show_split_preview()
        geo.build()
        geo.name_solids()

        geo.generate_mesh(maxh=0.03) # after naming solids, must generate mesh but avoid rebuilding
        return geo
    def cwg(self, maxh=0.04):
        geo = STEPImporter(r"circular_waveguide.step", auto_build=False)
        geo.build()
        geo.name_solids()
        geo.generate_mesh(maxh)
        return geo
    def cwg_split(self, maxh=0.04):
        geo = STEPImporter(r"circular_waveguide.step", auto_build=False)
        geo.add_splitting_plane_at_z(0.075)
        geo.add_splitting_plane_at_z(0.15)
        geo.split()

        geo.show_split_preview()
        geo.build()
        geo.name_solids()

        geo.generate_mesh(maxh=0.04) # after naming solids, must generate mesh but avoid rebuilding
        return geo
        
    def pillbox(self, maxh=0.1):
        geo = STEPImporter(r"../pillbox/pillbox.step", auto_build=False)
        geo.build()
        geo.define_ports(xmin='port1', xmax='port2')
        geo.generate_mesh(maxh=maxh) # after naming solids, must generate mesh but avoid rebuilding
        return geo
        
    def pillbox_split(self, maxh=0.04):
        geo = STEPImporter(r"../pillbox/pillbox.step", auto_build=False)
        geo.add_splitting_plane_at_x(0.0)
        geo.split()

        geo.show_split_preview()
        geo.build()
        geo.name_solids(sort_axis='X')

        geo.generate_mesh(maxh=maxh) # after naming solids, must generate mesh but avoid rebuilding
        return geo

In [ ]:
models = Models()
geo = models.pillbox_split()
print(geo.mesh.GetMaterials())
geo.show('mesh')

In [ ]:
from ngsolve import BoundaryFromVolumeCF
# Create a field that distinguishes regions (values don't matter, just need to be different)
mat_cf = geo.mesh.MaterialCF({"cell_1": 1, "cell_2": 2})

# Draw it
Draw(BoundaryFromVolumeCF(mat_cf), geo.mesh, "Geometry", draw_vol=True, draw_surf=True)
geo.mesh.GetMaterials()

In [ ]:
fmin, fmax, nsamples = 0.001, 2, 100
fds = FrequencyDomainSolver(geo, order=3)
fds.save(r"C:\Users\Soske\Documents\git_projects\cavsim3d_simulations\pillbox")

In [ ]:
%time fds.assemble_matrices(nportmodes=3)
%time result = fds.solve(fmin, fmax, nsamples)

In [ ]:
fds.plot_port_mode('port2', 0)

In [ ]:
fds.plot_field(20, domain='cell_2')

In [ ]:
#import cst result
cstresult = CSTResult(r'C:\Users\Soske\Documents\CST\pillbox')

fig, ax = fds.foms.plot_s(['1(1)2(1)'])
fig, ax = cstresult.plot_s(['1(1)2(1)'], ax=ax)
plt.show()

In [ ]:
roms = fds.foms.reduce()
roms

In [ ]:
%time roms_concat = roms.concatenate()
print()
%time result = roms_concat.solve(fmin, fmax, 1000)
print()
%time roms_concat_rom = roms_concat.reduce()
print()
%time result = roms_concat_rom.solve(fmin, fmax, 1000)
print()

# rom_result = roms.solve(fmin, fmax, 1000)

In [ ]:
# cstresult = CSTResult(r'C:\Users\Soske\Documents\CST\pillbox')

# from analytical.circular_waveguide import CWGAnalytical
# radius = 150e-3  # Width: 100 mm
# L = 300e-3  # Length: 200 mm
# cstresult = CWGAnalytical(radius=radius, length=L)
which = ['1(1)1(1)']
fig, axs = plt.subplot_mosaic([[1], [3]], layout='constrained', figsize=(6,8), sharex=True)

cstresult.plot_s(['1(1)1(1)'], ax=axs[1], linewidth=2, label='CST Studio')
roms_concat.plot_s(['1(1)1(1)'], ax=axs[1], label='Concatenated Reduced System')
# roms_concat_rom.plot_s(['1(1)1(1)'], ax=axs[1])
axs[1].set_ylabel(r'$|$ S1(1)1(1) $|$ [dB]')
axs[1].set_title(r'')
axs[1].set_xlabel(r'')

cstresult.plot_s(['1(1)1(1)'], ax=axs[3], linewidth=2, plot_type='phase', label='CST Studio')
roms_concat.plot_s(['1(1)1(1)'], ax=axs[3], plot_type='phase', label='Concatenated Reduced System')
# roms_concat_rom.plot_s(['1(1)1(1)'], ax=axs[3], plot_type='phase')
axs[3].set_ylabel(r'$\angle$ S1(1)1(1) [$^\circ$]')
axs[3].set_title(r'')

# plt.savefig('pillbox_sparameter.png', dpi=300)
plt.show()

In [ ]:
import matplotlib as mpl
def plot_settings():
    mpl.rcParams["font.family"] = "cmr10"
    mpl.rcParams['text.usetex'] = False  # Disable LaTeX rendering to use mathtext
    # mpl.rcParams['mathtext.fontset'] = 'cm'
    mpl.rcParams['axes.formatter.use_mathtext'] = True
    mpl.rcParams['xtick.labelsize'] = 18
    mpl.rcParams['ytick.labelsize'] = 18

    mpl.rcParams['axes.labelsize'] = 18
    mpl.rcParams['axes.titlesize'] = 18
    mpl.rcParams['legend.fontsize'] = 12
    mpl.rcParams['legend.title_fontsize'] = 18

    mpl.rcParams['figure.figsize'] = [12, 6]
    mpl.rcParams['figure.dpi'] = 100

    # Set the desired colormap
    plt.rcParams['axes.prop_cycle'] = plt.cycler('color', plt.cm.Set1.colors)
    
plot_settings()

In [ ]:
roms_concat.plot_field(897)

In [ ]:
# fds.foms.roms

In [ ]:
fds.save(r"C:\Users\Soske\Documents\git_projects\cavsim3d_simulations\pillbox")

In [1]:
# =============================================================================
# Example 1: Basic Sequential Assembly
# =============================================================================
from utils.visualization import *
from geometry.importers import STEPImporter
from geometry.assembly import Assembly
from geometry.primitives import RectangularWaveguide, CircularWaveguide
from geometry.component_registry import get_global_cache, ComputeMethod
from utils.visualization import (
    plot_z_comparison, 
    plot_s_comparison, 
    plot_all_parameters,
    plot_eigenfrequencies,
    ParameterPlotter,
    EigenfrequencyPlotter
)

# Create waveguide sections
wg1 = RectangularWaveguide(a=0.1, L=0.3, maxh=0.02)
wg2 = RectangularWaveguide(a=0.1, L=0.5, maxh=0.02)
wg3 = RectangularWaveguide(a=0.1, L=0.2, maxh=0.02)

# Check tags - wg1 and wg3 have same geometry params (different L though)
print(f"wg1 tag: {wg1.tag}")
print(f"wg2 tag: {wg2.tag}")

# Build assembly
assembly = Assembly(main_axis='Z')
assembly.add("input", wg1)
assembly.add("middle", wg2, after="input")
assembly.add("output", wg3, after="middle")

# Preview before building
assembly.inspect()

# Build and mesh
assembly.build()
assembly.generate_mesh(maxh=0.02)
assembly.show("mesh")

# Check assembly info
assembly.print_info()


# # =============================================================================
# # Example 2: Mosaic Layout (T-Junction)
# # =============================================================================

# # Load T-junction from STEP file
# t_junction = STEPImporter("t_junction.step", unit='mm', auto_build=True)

# # Create waveguide branches
# wg_horizontal = RectangularWaveguide(a=0.1, L=0.4, maxh=0.02)
# wg_vertical = RectangularWaveguide(a=0.1, L=0.3, maxh=0.02)

# # Define layout using mosaic
# layout = '''
#     .     top      .
#   left   center  right
# '''

# assembly = Assembly.from_mosaic(
#     layout,
#     components={
#         'center': t_junction,
#         'top': wg_vertical,
#         'left': wg_horizontal,
#         'right': wg_horizontal  # Same object used twice!
#     },
#     grid_plane='XY',
#     spacing=0.0
# )

# # Note: 'left' and 'right' share the same geometry object
# # The solver can compute once and reuse the solution
# unique = assembly.get_unique_components()
# print(f"Unique geometries: {len(unique)}")  # Will be 3, not 4

# assembly.build()
# assembly.show()




wg1 tag: RectangularWaveguide:83433ab2
wg2 tag: RectangularWaveguide:4b55a3bf


WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': 3…


Assembly: 3 components
------------------------------------------------------------
  [0] input: RectangularWaveguide
      Position: (0.0000, 0.0000, 0.0000)
      Size:     (0.1000, 0.0500, 0.3000)
  [1] middle: RectangularWaveguide
      Position: (0.0000, 0.0000, 0.3000)
      Size:     (0.1000, 0.0500, 0.5000)
  [2] output: RectangularWaveguide
      Position: (0.0000, 0.0000, 0.8000)
      Size:     (0.1000, 0.0500, 0.2000)
Port naming complete:
  Total ports: 4
  External ports: 2
  Interface ports: 2


WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.25…


ASSEMBLY INFORMATION
Total components:       3
Unique geometries:      3
Identical groups:       0
Main axis:              Z
Built:                  True
Has mesh:               True

Ports:
  Total:                4
  External:             2
  Interfaces:           2

Bounding Box:
  Min: (-0.0000, -0.0000, -0.0000)
  Max: (0.1000, 0.0500, 1.0000)

Components:
  [0] input: RectangularWaveguide
       Position: (0.0000, 0.0000, 0.0000)
  [1] middle: RectangularWaveguide
       Position: (0.0000, 0.0000, 0.3000)
  [2] output: RectangularWaveguide
       Position: (0.0000, 0.0000, 0.8000)

ASSEMBLY PORT MAP
Main axis: Z
Total ports: 4
----------------------------------------------------------------------
  port1    │ Z=+0.000000 │ EXTERNAL   │ 1 face(s) ◀▶
           │ Components: input.port1
  port2    │ Z=+0.300000 │ INTERFACE  │ 2 face(s) ◀┃▶
           │ Components: input.port2, middle.port1
  port3    │ Z=+0.800000 │ INTERFACE  │ 2 face(s) ◀┃▶
           │ Components: middle.port2,

In [2]:
# =============================================================================
# Example 3: Nested Assemblies (Sub-assemblies)
# =============================================================================
from pathlib import Path

# Create a reusable "taper section" sub-assembly
def create_taper_section(a_in, a_out, length, n_steps=3):
    """Create a stepped taper as a sub-assembly."""
    taper = Assembly(main_axis='Z')
    
    step_length = length / n_steps
    for i in range(n_steps):
        # Linear interpolation of width
        a = a_in + (a_out - a_in) * (i + 0.5) / n_steps
        wg = RectangularWaveguide(a=a, L=step_length, maxh=0.01)
        
        if i == 0:
            taper.add(f"step_{i}", wg)
        else:
            taper.add(f"step_{i}", wg, after=f"step_{i-1}")
    
    taper.build()
    return taper

# Create components
input_wg = RectangularWaveguide(a=0.08, L=0.2, maxh=0.02)
taper_in = create_taper_section(a_in=0.08, a_out=0.12, length=0.15)
main_section = RectangularWaveguide(a=0.12, L=0.5, maxh=0.02)
taper_out = create_taper_section(a_in=0.12, a_out=0.08, length=0.15)
output_wg = RectangularWaveguide(a=0.08, L=0.2, maxh=0.02)

# Build main assembly with sub-assemblies
main_assembly = Assembly(main_axis='Z')
main_assembly.add("input", input_wg)
main_assembly.add("taper_in", taper_in, after="input")      # Sub-assembly!
main_assembly.add("main", main_section, after="taper_in")
main_assembly.add("taper_out", taper_out, after="main")     # Sub-assembly!
main_assembly.add("output", output_wg, after="taper_out")

# Inspect shows sub-assemblies
main_assembly.inspect()

# Build flattens everything
main_assembly.build()
main_assembly.generate_mesh(maxh=0.02)

# Get flattened view for solver
flat = main_assembly.flatten()
print(f"\nFlattened assembly has {len(flat)} geometries:")
for key, geo, transform in flat:
    print(f"  {key}: {type(geo).__name__} at {transform.translation}")


# =============================================================================
# Example 4: Tagging and Solution Caching
# =============================================================================

from geometry.component_registry import get_global_cache, SolutionCache

# Setup persistent cache
cache = SolutionCache(max_size=500, persist_path=Path("./solution_cache.pkl"))

# Create identical waveguides (same parameters = same tag)
wg_a = RectangularWaveguide(a=0.1, L=0.3, maxh=0.02)
wg_b = RectangularWaveguide(a=0.1, L=0.3, maxh=0.02)  # Same params!
wg_c = RectangularWaveguide(a=0.1, L=0.5, maxh=0.02)  # Different L

# Check tag equality
print(f"wg_a tag: {wg_a.tag}")
print(f"wg_b tag: {wg_b.tag}")
print(f"Tags match: {wg_a.tag.full_hash == wg_b.tag.full_hash}")  # True!

# Simulate solving wg_a and caching result
fake_solution = {"S11": 0.1, "S21": 0.9}  # Placeholder
wg_a.cache_solution(fake_solution, frequency=10e9)

# Now wg_b can use the cached solution (same tag)
if wg_b.has_cached_solution():
    cached = wg_b.get_cached_solution()
    print(f"Retrieved cached solution: {cached.solution_data}")
else:
    print("No cached solution (need to compute)")

# Build assembly and check what needs computation
assembly = Assembly()
assembly.add("seg_a", wg_a)
assembly.add("seg_b", wg_b, after="seg_a")
assembly.add("seg_c", wg_c, after="seg_b")

uncached = assembly.get_uncached_components()
print(f"Components needing computation: {uncached}")  # Only 'seg_c'

# Unique geometries
unique = assembly.get_unique_components()
print(f"Unique geometries: {assembly.count_unique_geometries()}")  # 2


# =============================================================================
# Example 5: Analytical vs Numeric Computation
# =============================================================================

# Create waveguides with different compute methods
wg_numeric = RectangularWaveguide(
    a=0.1, L=0.3, maxh=0.02,
    compute_method='numeric'
)

wg_analytical = RectangularWaveguide(
    a=0.1, L=0.3, maxh=0.02,
    compute_method='analytical'
)

# Check properties
print(f"Numeric method: {wg_numeric.compute_method}")
print(f"Analytical method: {wg_analytical.compute_method}")
print(f"Supports analytical: {wg_analytical.supports_analytical}")

# Get analytical mode info (available for analytical method)
if wg_analytical.supports_analytical:
    modes = wg_analytical.get_analytical_modes(n_modes=5)
    print("\nFirst 5 modes:")
    for mode in modes:
        fc_ghz = mode['cutoff_frequency'] / 1e9
        print(f"  {mode['type']}{mode['indices']}: fc = {fc_ghz:.3f} GHz")

# Solver can check compute_method and dispatch accordingly
def solve_component(component):
    """Example solver dispatch based on compute method."""
    if component.compute_method == ComputeMethod.ANALYTICAL:
        if component.supports_analytical:
            print(f"Using analytical solution for {type(component).__name__}")
            # return analytical_solver.solve(component)
        else:
            print(f"Analytical not supported, falling back to numeric")
            # return numeric_solver.solve(component)
    else:
        print(f"Using numeric FEM solution for {type(component).__name__}")
        # return numeric_solver.solve(component)

solve_component(wg_numeric)
solve_component(wg_analytical)


# =============================================================================
# Example 6: Complex Multi-Cell Cavity Assembly
# =============================================================================

# Load cavity cells (could be from STEP files or analytical)
cells = []
for i in range(9):
    # In practice, these might be STEPImporter objects
    # Here we use circular waveguides as placeholders
    cell = CircularWaveguide(
        radius=0.05,
        length=0.1,
        maxh=0.02,
        compute_method='numeric'
    )
    cell.custom_tag = f"TESLA_cell_{i+1}"
    cells.append(cell)

# Create cavity assembly
cavity = Assembly(main_axis='Z')

# Add beam pipes
input_pipe = CircularWaveguide(radius=0.02, length=0.15, maxh=0.01)
output_pipe = CircularWaveguide(radius=0.02, length=0.15, maxh=0.01)

cavity.add("input_pipe", input_pipe)

# Add cells
for i, cell in enumerate(cells):
    if i == 0:
        cavity.add(f"cell_{i+1}", cell, after="input_pipe")
    else:
        cavity.add(f"cell_{i+1}", cell, after=f"cell_{i}")

cavity.add("output_pipe", output_pipe, after="cell_9")

# Print info - shows unique geometries and cache status
cavity.print_info()

# Since all cells are identical (same radius, length), they share a tag
unique = cavity.get_unique_components()
print(f"\nUnique geometries: {len(unique)}")
print("This means we only need to solve {len(unique)} problems, not {len(cavity)}!")

# Build
cavity.build()
cavity.generate_mesh(maxh=0.02)
cavity.show()


# =============================================================================
# Example 7: Saving and Loading Cache
# =============================================================================

from geometry.component_registry import get_global_cache, set_global_cache, SolutionCache
from pathlib import Path

# Create persistent cache
cache = SolutionCache(
    max_size=1000,
    persist_path=Path("./em_cache/solutions.pkl")
)
set_global_cache(cache)

# ... do computations ...

# Save cache to disk
cache.save()
print(f"Cache saved: {cache.stats()}")

# Later, in a new session:
cache2 = SolutionCache(persist_path=Path("./em_cache/solutions.pkl"))
set_global_cache(cache2)
print(f"Cache loaded: {cache2.stats()}")


# =============================================================================
# Example 8: Finding Reusable Solutions Across Projects
# =============================================================================

# The tagging system allows finding matching solutions even if
# the component was created in a different assembly or project

wg1 = RectangularWaveguide(a=0.1, L=0.3, maxh=0.02)
print(f"Looking for solutions matching: {wg1.tag}")

# Find all cached solutions that match this geometry
# (maybe with different mesh parameters)
matches = get_global_cache().find_matching(wg1.tag, ignore_mesh=True)
print(f"Found {len(matches)} matching solutions (any mesh):")
for match in matches:
    print(f"  - Mesh hash: {match.tag.mesh_hash[:8]}...")
    print(f"    Computed: {match.created_at}")
    print(f"    Method: {match.compute_method}")

Port naming complete:
  Total ports: 4
  External ports: 2
  Interface ports: 2
Port naming complete:
  Total ports: 4
  External ports: 2
  Interface ports: 2


WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': 3…


Assembly: 5 components
------------------------------------------------------------
  [0] input: RectangularWaveguide
      Position: (0.0000, 0.0000, 0.0000)
      Size:     (0.0800, 0.0400, 0.2000)
  [1] taper_in: Assembly [sub-assembly]
      Position: (-0.0033, -0.0017, 0.2000)
      Size:     (0.1133, 0.0567, 0.1500)
  [2] main: RectangularWaveguide
      Position: (-0.0200, -0.0100, 0.3500)
      Size:     (0.1200, 0.0600, 0.5000)
  [3] taper_out: Assembly [sub-assembly]
      Position: (-0.0167, -0.0083, 0.8500)
      Size:     (0.1133, 0.0567, 0.1500)
  [4] output: RectangularWaveguide
      Position: (0.0000, 0.0000, 1.0000)
      Size:     (0.0800, 0.0400, 0.2000)
Port naming complete:
  Total ports: 10
  External ports: 2
  Interface ports: 8


AttributeError: 'Assembly' object has no attribute 'flatten'

In [4]:
from core.em_project import EMProject
from geometry.primitives import RectangularWaveguide, CircularWaveguide

# 1. Start project
proj = EMProject(name='Sequential_Assembly', base_dir='./simulations')

# 2. Create assembly
assembly = proj.create_assembly(main_axis='Z')

# 3. Add components
wg1 = RectangularWaveguide(a=0.1, L=0.3)
wg2 = RectangularWaveguide(a=0.1, L=0.3)

assembly.add("input", wg1)
assembly.add("output", wg2, after="input")

# 4. Build and save
assembly.build()
assembly.generate_mesh(maxh=0.02)
assembly.show('mesh')

Project 'Sequential_Assembly' exists. Loading automatically...
Saving project to simulations\Sequential_Assembly
Port naming complete:
  Total ports: 3
  External ports: 2
  Interface ports: 1


WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.25…